# MeSH KG Construction

This notebook walks through **Phase 1** of the unified KG pipeline:

| Step | Task |
|------|------|
| **1** | Fetch MeSH annotations for every PMID via PubMed E-utilities |
| **2** | Download & parse the MeSH descriptor hierarchy (`desc2026.xml`) |

Outputs land in `data/mesh_kg/`:
```
mesh_annotations.json          {pmid: [mesh_term, ...]}
mesh_descriptors.parquet       ui, name, tree_numbers, synonyms
mesh_kg_nodes.parquet          node_id, name, node_type, source
mesh_kg_edges_hierarchy.parquet  narrower_term_of triples
```

---
**Before running at scale:**  get a free NCBI API key at
https://www.ncbi.nlm.nih.gov/account/ and paste it in the `API_KEY` cell.
Without one the rate limit is 3 req/s (works but is ~3× slower).

In [1]:
import sys
sys.path.insert(0, "../src")  # make sure neurovlm is importable

import json
import pandas as pd
from pathlib import Path

from neurovlm.gnn.mesh import (
    fetch_mesh_for_pmids,
    load_annotations,
    check_mesh_coverage,
    annotations_to_dataframe,
    parse_mesh_descriptor_xml,
    build_mesh_hierarchy_triples,
    load_failed_pmids,
)

OUT = Path("data/mesh_kg")
OUT.mkdir(parents=True, exist_ok=True)

# ---- NCBI API key ----
# Get yours free at https://www.ncbi.nlm.nih.gov/account/
# Without one: 3 req/s limit.  With one: 10 req/s.
API_KEY = ""   # e.g. "abc123def456"

## Load PMIDs from existing dataset

The NLP KG was built from 1.2 M PubMed papers.  We need their PMIDs.
If your dataset already has MeSH metadata, check for it first — you may
only need to fill gaps rather than re-fetching everything.

**Adjust the cell below** to load PMIDs from wherever your paper metadata lives
(e.g. a parquet/CSV file produced during data collection).

In [2]:
# --- Load your PMID list here ---
# Example: if you have a parquet file with a 'pmid' column:
#
#   papers = pd.read_parquet("data/pubmed_papers.parquet")
#   pmids = papers["pmid"].astype(str).unique().tolist()
#
# For demonstration we create a small synthetic list.
# Replace with your real PMID source before running at scale.

PMID_SOURCE = "data/pubmed_neuroimaging.parquet"   # set to a Path/string pointing at your papers file

if PMID_SOURCE is not None:
    papers = pd.read_parquet(PMID_SOURCE)
    pmids = papers["pmid"].astype(str).unique().tolist()
else:
    # small demo set — remove this branch when running for real
    pmids = [
        "33483671", "34349469", "32719077", "30792285",
        "29184211", "28728020", "27642368", "25392494",
    ]

print(f"Total PMIDs to annotate: {len(pmids):,}")

Total PMIDs to annotate: 1,231,613


In [3]:
papers.columns

Index(['pmid', 'doi', 'year', 'journal', 'title', 'abstract', 'authors'], dtype='str')

### Check whether MeSH terms has already been fetched

In [4]:
from pathlib import Path

mesh_annot_path = OUT / "mesh_annotations_long.parquet"
if mesh_annot_path.exists():
    annot_df = pd.read_parquet(mesh_annot_path)
    print("Skip Step 1: mesh annotations already fetched and loaded.")
else:
    print("mesh_annotations_long.parquet not found. Proceed to Step 1 to fetch MeSH annotations.")

Skip Step 1: mesh annotations already fetched and loaded.


---
## Step 1 — Fetch MeSH annotations via PubMed E-utilities

Calls `efetch` in batches of 200 PMIDs.  Results are checkpointed to
`data/mesh_kg/mesh_annotations.json` every 500 batches so the run
can be safely interrupted and resumed.

**Estimated runtime** (1.2 M papers):
- With API key (10 req/s) : ~10 minutes
- Without API key (3 req/s): ~33 minutes

In [8]:
ANNOT_PATH = OUT / "mesh_annotations.json"

annotations = fetch_mesh_for_pmids(
    pmids,
    api_key=API_KEY,
    batch_size=200,
    out_path=ANNOT_PATH,
    checkpoint_every=500,
    resume=True,            # pick up where we left off if interrupted
)

print(f"\nAnnotations loaded: {len(annotations):,} PMIDs")

In [ ]:
from neurovlm.gnn.mesh import load_failed_pmids

failed_pmids = load_failed_pmids(ANNOT_PATH)
stats = check_mesh_coverage(annotations, pmids, failed_pmids=failed_pmids or None)
stats

> **Expected**: ≥ 90 % of indexed PubMed papers have MeSH terms.  Papers
> published in the last 6–12 months may not yet be fully annotated by NLM.

### Inspect sample annotations

In [13]:
sample_pmid = next(p for p in pmids if annotations.get(p))
print(f"PMID {sample_pmid}:")
for t in annotations[sample_pmid]:
    print(f"  {t}")

PMID 41747334:
  Humans
  Ischemic Stroke/diagnosis
  Diagnosis, Differential
  Male
  Fatal Outcome
  Aspergillosis/diagnosis
  Aspergillosis/pathology
  Aspergillosis/drug therapy
  Diffusion Magnetic Resonance Imaging
  Antifungal Agents/therapeutic use
  Middle Aged
  Immunocompromised Host
  Brain/pathology
  Brain/diagnostic imaging
  Aged
  Neuroaspergillosis/diagnosis
  Neuroaspergillosis/pathology


In [14]:
annot_df = annotations_to_dataframe(annotations)
print(f"Long-form rows: {len(annot_df):,}")
annot_df.head()

Long-form rows: 14,470,827


,pmid,mesh_term
0,41747334,Humans
1,41747334,Ischemic Stroke/diagnosis
2,41747334,"Diagnosis, Differential"
3,41747334,Male
4,41747334,Fatal Outcome


In [15]:
# Top MeSH terms by paper count
annot_df["mesh_term"].value_counts().head(20)

mesh_term
Humans                                909559
Male                                  537472
Female                                513754
Magnetic Resonance Imaging            356178
Adult                                 343009
Middle Aged                           303524
Aged                                  209891
Magnetic Resonance Imaging/methods    151208
Animals                               127273
Adolescent                            112025
Young Adult                           104944
Retrospective Studies                  97278
Tomography, X-Ray Computed             84838
Child                                  84405
Treatment Outcome                      75160
Brain Mapping                          66895
Aged, 80 and over                      65632
Brain/diagnostic imaging               61361
Diagnosis, Differential                56701
Electroencephalography                 54027
Name: count, dtype: int64

In [16]:
annot_df.to_parquet(OUT / "mesh_annotations_long.parquet", index=False)
print("Saved:", OUT / "mesh_annotations_long.parquet")

Saved: data/mesh_kg/mesh_annotations_long.parquet


---
## Step 2 — Download & parse the MeSH hierarchy

The full MeSH ontology ships as an annual XML release.  Download the
current year's descriptor file manually before running this section:

```bash
cd experiments/data/mesh_kg
curl -O https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml
```

The file is ~300 MB uncompressed.  The parser extracts for each descriptor:
- **DescriptorUI** — canonical MeSH ID (used as `node_id` in the KG)
- **DescriptorName** — canonical term name
- **TreeNumberList** — hierarchy codes (e.g. `A08.186.211`)
- **ConceptList** — synonyms and entry terms

From tree numbers the code derives:
```
(hippocampus, narrower_term_of, limbic system)
(limbic system, narrower_term_of, brain)
```

In [5]:
DESCRIPTOR_XML = OUT / "desc2026.xml"

if not DESCRIPTOR_XML.exists():
    print(
        f"desc2026.xml not found at {DESCRIPTOR_XML}.\n"
        "Download it with:\n"
        "  curl -O https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc2026.xml\n"
        "then place it in experiments/data/mesh_kg/ and rerun this cell."
    )
else:
    print(f"Found: {DESCRIPTOR_XML} ({DESCRIPTOR_XML.stat().st_size / 1e6:.0f} MB)")

Found: data/mesh_kg/desc2026.xml (313 MB)


In [6]:
if not DESCRIPTOR_XML.exists():
    print("Skipping — descriptor XML not yet downloaded.")
else:
    desc_df = parse_mesh_descriptor_xml(DESCRIPTOR_XML)
    print(f"Descriptors parsed: {len(desc_df):,}")
    desc_df.head(3)

Descriptors parsed: 31,110


In [7]:
if not DESCRIPTOR_XML.exists():
    print("Skipping.")
else:
    # Show descriptors with neuro-relevant tree numbers
    from neurovlm.gnn.mesh import NEURO_TREE_PREFIXES

    neuro_mask = desc_df["tree_numbers"].apply(
        lambda tns: any(tn.split(".")[0][:3] in NEURO_TREE_PREFIXES for tn in tns)
    )
    print(f"Neuro-relevant descriptors: {neuro_mask.sum():,} / {len(desc_df):,}")
    desc_df[neuro_mask].head(5)

Neuro-relevant descriptors: 9,284 / 31,110


In [8]:
if not DESCRIPTOR_XML.exists():
    print("Skipping.")
else:
    hierarchy_triples = build_mesh_hierarchy_triples(desc_df)
    print(f"Hierarchy triples: {len(hierarchy_triples):,}")
    hierarchy_triples.head(5)

Hierarchy triples: 42,519


## Export MeSH KG in unified schema

In [9]:
if not DESCRIPTOR_XML.exists():
    print("Skipping node export — descriptor XML not downloaded.")
else:
    def _infer_node_type(tree_numbers: list[str]) -> str:
        """Map MeSH tree numbers to a semantic type.

        Checks ALL tree numbers and returns the highest-priority match so that
        a descriptor with both A08.* and D12.* tree numbers is correctly labelled
        anatomical_region rather than molecular.

        Priority ladder
        ---------------
        1. A08/A09/A10/A11  → anatomical_region  (nervous system, sense organs, tissue, cells)
        2. any A            → anatomical_region  (all other anatomy: heart, lung, gut, …)
        3. F01/F02/F04      → cognitive_construct (behaviour, psych phenomena, behavioral disciplines)
        4. F03 / any C      → disorder           (mental disorders, all disease categories)
        5. any D            → molecular          (proteins, organic/inorganic chem, enzymes,
                                                   lipids, nucleic acids, pharmacological actions, …)
        6. E01-E07          → method             (diagnosis, therapeutics, surgery, investigative
                                                   techniques, equipment)
        7. any G            → biological_process (biological phenomena, physiology, genetics)
        8. any B            → organism           (bacteria, viruses, eukaryota, fungi)
        9. M01              → demographic        (persons: Humans, Male, Female, Aged, …)
        10. (anything else) → other              (geography Z, social sciences I, health admin N,
                                                   disciplines H, industry J, humanities K/L/V)
        """
        prefixes = {tn.split(".")[0] for tn in tree_numbers}
        top     = {tn[:1] for tn in prefixes}   # single-letter category

        # 1 & 2 — Anatomy (neuro-specific first so priority comment stays accurate)
        if prefixes & {"A08", "A09", "A10", "A11"}:
            return "anatomical_region"
        if "A" in top:
            return "anatomical_region"

        # 3 — Behaviour & psychological phenomena
        if prefixes & {"F01", "F02", "F04"}:
            return "cognitive_construct"

        # 4 — Disorders (psychiatric F03 and all disease C categories)
        if "F03" in prefixes or "C" in top:
            return "disorder"

        # 5 — Chemicals & drugs (all D sub-trees: proteins, enzymes, lipids,
        #      nucleic acids, inorganic/organic chem, pharmacological actions, …)
        if "D" in top:
            return "molecular"

        # 6 — Techniques & equipment
        if "E" in top:
            return "method"

        # 7 — Biological phenomena and processes
        if "G" in top:
            return "biological_process"

        # 8 — Organisms
        if "B" in top:
            return "organism"

        # 9 — Demographic groups (M01: Persons — Humans, Male, Female, Aged, …)
        if "M" in top:
            return "demographic"

        # 10 — Legitimate residual: geography (Z), social sciences (I),
        #       health administration (N), disciplines (H), industry (J),
        #       humanities/information science (K/L), publication types (V)
        return "other"

    mesh_kg_nodes = pd.DataFrame({
        "node_id"   : desc_df["ui"],
        "name"      : desc_df["name"],
        "node_type" : desc_df["tree_numbers"].apply(_infer_node_type),
        "source"    : "mesh",
    }).reset_index(drop=True)

    mesh_kg_nodes.to_parquet(OUT / "mesh_kg_nodes.parquet", index=False)
    desc_df.to_parquet(OUT / "mesh_descriptors.parquet", index=False)

    print(f"Saved {len(mesh_kg_nodes):,} nodes → {OUT / 'mesh_kg_nodes.parquet'}")
    print(mesh_kg_nodes["node_type"].value_counts())


Saved 31,110 nodes → data/mesh_kg/mesh_kg_nodes.parquet
node_type
molecular              10671
disorder                5087
organism                3938
other                   3196
method                  3023
anatomical_region       1908
biological_process      1886
cognitive_construct     1056
demographic              345
Name: count, dtype: int64


In [10]:
if not DESCRIPTOR_XML.exists():
    print("Skipping.")
else:
    # ── How node types were determined ────────────────────────────────────────
    # _infer_node_type walks a fixed priority ladder over the 3-char tree-number
    # prefixes of each descriptor.  The ladder is:
    #
    #   A08/A09/A10/A11  → anatomical_region   (nervous system, sense organs, tissue, cells)
    #   F01/F02          → cognitive_construct  (behaviour & psychological phenomena)
    #   F03 / any C      → disorder            (mental disorders, all disease categories)
    #   D12/D02/D03/D04/D09 → molecular        (proteins, organic/heterocyclic chem)
    #   E01/E02/E04/E05  → method              (diagnostic & investigative techniques)
    #   any G            → biological_process  (biological phenomena)
    #   (anything else)  → other
    #
    # Priority matters for multi-category descriptors: a descriptor with both
    # A08.* (nervous system anatomy) and D12.* (proteins) tree numbers is
    # labelled anatomical_region, not molecular — A08 wins.
    #
    # The 1,130 anatomical_region count covers descriptors that have at least
    # one A08/A09/A10/A11 tree number.  The remaining ~8,154 neuro-relevant
    # descriptors carry F01/F02/F03/C10/C23/D12/D02 tree numbers without A08,
    # so they fall through to cognitive_construct, disorder, or molecular.

    neuro_node_ids = set(desc_df.loc[neuro_mask, "ui"])
    neuro_nodes    = mesh_kg_nodes[mesh_kg_nodes["node_id"].isin(neuro_node_ids)]

    print(f"Neuro-relevant descriptors : {len(neuro_nodes):,} / {len(mesh_kg_nodes):,}")
    print()
    print("Node-type breakdown for neuro-relevant descriptors:")
    print(neuro_nodes["node_type"].value_counts().to_string())
    print()
    # Spot-check: confirm pure A08 descriptors (no other category) → anatomical_region
    pure_a08 = desc_df[desc_df["tree_numbers"].apply(
        lambda tns: any(tn.startswith("A08") for tn in tns)
        and not any(tn[:3] in {"D12","D02","D03","D04"} for tn in tns)
    )]
    print(f"Descriptors with A08.* tree numbers : {desc_df['tree_numbers'].apply(lambda tns: any(t.startswith('A08') for t in tns)).sum():,}")
    print(f"  → all labelled anatomical_region  : {(mesh_kg_nodes[mesh_kg_nodes['node_id'].isin(desc_df[desc_df['tree_numbers'].apply(lambda tns: any(t.startswith('A08') for t in tns))]['ui'])]['node_type'] == 'anatomical_region').all()}")


Neuro-relevant descriptors : 9,284 / 31,110

Node-type breakdown for neuro-relevant descriptors:
node_type
molecular              6269
disorder               1711
cognitive_construct     874
anatomical_region       430

Descriptors with A08.* tree numbers : 404
  → all labelled anatomical_region  : True


In [11]:
if not DESCRIPTOR_XML.exists():
    print("Skipping edge export.")
else:
    hierarchy_triples.to_parquet(OUT / "mesh_kg_edges_hierarchy.parquet", index=False)
    print(f"Saved {len(hierarchy_triples):,} hierarchy triples → {OUT / 'mesh_kg_edges_hierarchy.parquet'}")
    print()
    print("Relation type breakdown:")
    print(hierarchy_triples["relation_type"].value_counts())

Saved 42,519 hierarchy triples → data/mesh_kg/mesh_kg_edges_hierarchy.parquet

Relation type breakdown:
relation_type
narrower_term_of    42519
Name: count, dtype: int64


## Summary

| File | Contents |
|---|---|
| `mesh_annotations.json` | `{pmid: [mesh_term, ...]}` for all fetched papers |
| `mesh_annotations_long.parquet` | long-form `(pmid, mesh_term)` table |
| `mesh_descriptors.parquet` | full descriptor table with tree numbers & synonyms |
| `mesh_kg_nodes.parquet` | nodes in unified schema |
| `mesh_kg_edges_hierarchy.parquet` | `narrower_term_of` triples from hierarchy |

**Next steps (Steps 3 & 4)** are implemented in `mesh_kg_cooccurrence.ipynb`:
- Step 3: generate co-occurrence triples from the `mesh_annotations` + paper list
- Step 4: promote high-confidence co-occurrences to typed relations